<a href="https://colab.research.google.com/github/Zarrialvi/AI-PROJECTS/blob/main/CPUScheduling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
CPU Scheduling Simulator - Fixed Professional Edition
BS Software Engineering Project
All anomalies removed and improvements added
"""

import tkinter as tk
from tkinter import ttk, messagebox
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import copy

class Process:
    def __init__(self, pid, arrival, burst, priority):
        self.pid = pid
        self.arrival = arrival
        self.burst = burst
        self.priority = priority
        self.remaining = burst
        self.completion = 0
        self.turnaround = 0
        self.waiting = 0
        self.response = -1
        self.start_time = -1
        # Used for MLFQ tracking
        self.current_queue = 1
        self.service_time = 0

class CPUScheduler:
    def __init__(self, root):
        self.root = root
        self.root.title("🖥️ CPU Scheduling Simulator")
        self.root.geometry("1400x900")
        self.root.configure(bg="#1e293b")
        self.processes = []
        self.colors = ['#3b82f6', '#ef4444', '#10b981', '#f59e0b',
                       '#8b5cf6', '#ec4899', '#06b6d4', '#84cc16']
        self.setup_ui()

    def setup_ui(self):
        # --- (GUI setup remains largely the same for brevity) ---
        main_frame = tk.Frame(self.root, bg="#1e293b")
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        # Header
        header = tk.Frame(main_frame, bg="#334155", relief=tk.RAISED, bd=2)
        header.pack(fill=tk.X, pady=(0, 10))
        tk.Label(header, text="🖥️ CPU Scheduling Simulator",
                 font=("Helvetica", 24, "bold"), bg="#334155", fg="white").pack(pady=10)
        tk.Label(header, text="7 Algorithms | BS Software Engineering",
                 font=("Helvetica", 10), bg="#334155", fg="#94a3b8").pack(pady=(0, 10))

        content = tk.Frame(main_frame, bg="#1e293b")
        content.pack(fill=tk.BOTH, expand=True)

        # Left Panel
        left = tk.Frame(content, bg="#334155", relief=tk.RAISED, bd=2, width=350)
        left.pack(side=tk.LEFT, fill=tk.BOTH, padx=(0, 5))
        left.pack_propagate(False)

        tk.Label(left, text="⚙️ Configuration", font=("Helvetica", 16, "bold"),
                 bg="#334155", fg="white").pack(pady=10, padx=10, anchor=tk.W)

        # Algorithm Selection
        algo_frame = tk.Frame(left, bg="#334155")
        algo_frame.pack(fill=tk.X, padx=10, pady=5)
        tk.Label(algo_frame, text="Algorithm:", font=("Helvetica", 10, "bold"),
                 bg="#334155", fg="white").pack(anchor=tk.W)

        self.algorithm_var = tk.StringVar(value="FCFS")
        algos = ["FCFS", "SJF", "SRTF", "Round Robin",
                 "Priority (NP)", "Priority (P)", "MLQ", "MLFQ"]
        ttk.Combobox(algo_frame, textvariable=self.algorithm_var, values=algos,
                     state="readonly", width=35).pack(fill=tk.X, pady=5)

        # Quantum Input
        self.quantum_frame = tk.Frame(left, bg="#334155")
        self.quantum_frame.pack(fill=tk.X, padx=10, pady=5) # Keep packed by default
        tk.Label(self.quantum_frame, text="Time Quantum (Q1):",
                 font=("Helvetica", 10, "bold"), bg="#334155", fg="white").pack(anchor=tk.W)
        self.quantum_var = tk.StringVar(value="2")
        tk.Entry(self.quantum_frame, textvariable=self.quantum_var,
                 font=("Helvetica", 10), width=10).pack(anchor=tk.W, pady=5)

        # Process Input
        tk.Label(left, text="📋 Process Input", font=("Helvetica", 16, "bold"),
                 bg="#334155", fg="white").pack(pady=10, padx=10, anchor=tk.W)

        input_frame = tk.Frame(left, bg="#334155")
        input_frame.pack(fill=tk.X, padx=10, pady=5)

        self.entry_vars = {}
        for label, key, default in [("Arrival:", "arrival", "0"),
                                     ("Burst:", "burst", "1"),
                                     ("Priority:", "priority", "0")]:
            f = tk.Frame(input_frame, bg="#334155")
            f.pack(fill=tk.X, pady=3)
            tk.Label(f, text=label, font=("Helvetica", 9), bg="#334155",
                     fg="white", width=10, anchor=tk.W).pack(side=tk.LEFT)
            var = tk.StringVar(value=default)
            tk.Entry(f, textvariable=var, font=("Helvetica", 9), width=15).pack(side=tk.LEFT, padx=5)
            self.entry_vars[key] = var

        tk.Button(left, text="➕ Add Process", command=self.add_process,
                  bg="#10b981", fg="white", font=("Helvetica", 10, "bold"),
                  cursor="hand2").pack(pady=10, padx=10, fill=tk.X)

        # Process List
        tk.Label(left, text="📊 Process List", font=("Helvetica", 14, "bold"),
                 bg="#334155", fg="white").pack(pady=5, padx=10, anchor=tk.W)

        tree_frame = tk.Frame(left, bg="#334155")
        tree_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)

        self.process_tree = ttk.Treeview(tree_frame,
                                         columns=("PID", "Arrival", "Burst", "Priority"),
                                         show="headings", height=8)
        for col in ["PID", "Arrival", "Burst", "Priority"]:
            self.process_tree.heading(col, text=col)
            self.process_tree.column(col, width=70, anchor=tk.CENTER)

        scroll = ttk.Scrollbar(tree_frame, orient=tk.VERTICAL, command=self.process_tree.yview)
        self.process_tree.configure(yscrollcommand=scroll.set)
        self.process_tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scroll.pack(side=tk.RIGHT, fill=tk.Y)

        # Buttons
        btn_frame = tk.Frame(left, bg="#334155")
        btn_frame.pack(fill=tk.X, padx=10, pady=10)
        tk.Button(btn_frame, text="🗑️ Delete", command=self.delete_process,
                  bg="#ef4444", fg="white", font=("Helvetica", 9, "bold"),
                  cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True, padx=2)
        tk.Button(btn_frame, text="🔄 Clear", command=self.clear_all,
                  bg="#f59e0b", fg="white", font=("Helvetica", 9, "bold"),
                  cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True, padx=2)

        tk.Button(left, text="▶️ RUN SCHEDULER", command=self.run_scheduler,
                  bg="#3b82f6", fg="white", font=("Helvetica", 12, "bold"),
                  cursor="hand2", height=2).pack(pady=10, padx=10, fill=tk.X)

        # Right Panel
        self.right_panel = tk.Frame(content, bg="#334155", relief=tk.RAISED, bd=2)
        self.right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(5, 0))
        self.show_initial_message()
    # --- (Helper methods like show_initial_message, add_process, etc. omitted for brevity, they were fine) ---

    def show_initial_message(self):
        for w in self.right_panel.winfo_children():
            w.destroy()
        msg = tk.Frame(self.right_panel, bg="#334155")
        msg.place(relx=0.5, rely=0.5, anchor=tk.CENTER)
        tk.Label(msg, text="📊", font=("Helvetica", 64), bg="#334155", fg="#64748b").pack()
        tk.Label(msg, text="No Results Yet", font=("Helvetica", 20, "bold"),
                 bg="#334155", fg="white").pack(pady=10)
        tk.Label(msg, text="Add processes and click RUN", font=("Helvetica", 12),
                 bg="#334155", fg="#94a3b8").pack()

    def add_process(self):
        try:
            arrival = int(self.entry_vars["arrival"].get())
            burst = int(self.entry_vars["burst"].get())
            priority = int(self.entry_vars["priority"].get())

            if burst <= 0:
                messagebox.showerror("Error", "Burst time must be > 0!")
                return
            if arrival < 0 or priority < 0:
                messagebox.showerror("Error", "Values cannot be negative!")
                return

            pid = f"P{len(self.processes) + 1}"
            proc = Process(pid, arrival, burst, priority)
            self.processes.append(proc)
            self.process_tree.insert("", tk.END, values=(pid, arrival, burst, priority))
            self.entry_vars["burst"].set("1")
        except ValueError:
            messagebox.showerror("Error", "Enter valid integers!")

    def delete_process(self):
        sel = self.process_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a process to delete!")
            return
        idx = self.process_tree.index(sel[0])
        self.process_tree.delete(sel[0])
        del self.processes[idx]
        self.refresh_list()

    def clear_all(self):
        if self.processes and messagebox.askyesno("Confirm", "Delete all?"):
            self.processes.clear()
            for item in self.process_tree.get_children():
                self.process_tree.delete(item)

    def refresh_list(self):
        for item in self.process_tree.get_children():
            self.process_tree.delete(item)
        for i, p in enumerate(self.processes):
            p.pid = f"P{i + 1}"
            self.process_tree.insert("", tk.END, values=(p.pid, p.arrival, p.burst, p.priority))

    def run_scheduler(self):
        if not self.processes:
            messagebox.showwarning("Warning", "Add at least one process!")
            return

        algo = self.algorithm_var.get()
        quantum = 2 # Default

        # Only try to read quantum if needed for RR or MLFQ
        if algo in ["Round Robin", "MLFQ"]:
            try:
                q_val = int(self.quantum_var.get())
                if q_val > 0:
                    quantum = q_val
                else:
                    messagebox.showwarning("Warning", "Quantum must be > 0. Defaulting to 2.")
            except ValueError:
                messagebox.showwarning("Warning", "Invalid Quantum. Defaulting to 2.")

        try:
            if algo == "FCFS":
                gantt, metrics = self.fcfs()
            elif algo == "SJF":
                gantt, metrics = self.sjf()
            elif algo == "SRTF":
                gantt, metrics = self.srtf()
            elif algo == "Round Robin":
                gantt, metrics = self.round_robin(quantum)
            elif algo == "Priority (NP)":
                gantt, metrics = self.priority_np()
            elif algo == "Priority (P)":
                gantt, metrics = self.priority_p()
            elif algo == "MLQ":
                gantt, metrics = self.mlq()
            elif algo == "MLFQ":
                # Use quantum for Q1, Q2=2*Q1, Q3=4*Q1
                gantt, metrics = self.mlfq(quantum, quantum * 2, quantum * 4)
            self.display_results(gantt, metrics, algo)
        except Exception as e:
            # Catching generic errors might hide bugs, but for a simulator,
            # it prevents a crash from an unexpected state.
            import traceback
            messagebox.showerror("Error", f"Execution failed: {str(e)}\n\n{traceback.format_exc()}")

    # --- SCHEDULING ALGORITHMS ---

    def fcfs(self):
        # (FCFS logic was correct)
        procs = sorted(copy.deepcopy(self.processes), key=lambda x: x.arrival)
        gantt, time = [], 0
        for p in procs:
            time = max(time, p.arrival)
            p.start_time = time
            p.completion = time + p.burst
            p.turnaround = p.completion - p.arrival
            p.waiting = p.turnaround - p.burst
            p.response = p.start_time - p.arrival
            gantt.append((p.pid, time, p.completion))
            time = p.completion
        return gantt, procs

    def sjf(self):
        # (SJF logic was correct)
        procs = copy.deepcopy(self.processes)
        gantt, time, done = [], 0, []
        while len(done) < len(procs):
            # Sort by Burst Time, then by Arrival Time for tie-breaking
            avail = [p for p in procs if p.arrival <= time and p not in done]
            if not avail:
                # Find the arrival time of the next process and jump to it
                next_arrival = min(p.arrival for p in procs if p not in done)
                if next_arrival > time:
                    # Insert idle time
                    if gantt and gantt[-1][0] == "Idle": # Coalesce idle time
                        gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                    else:
                        gantt.append(("Idle", time, next_arrival))
                    time = next_arrival
                continue

            p = min(avail, key=lambda x: (x.burst, x.arrival))
            p.start_time = time
            p.completion = time + p.burst
            p.turnaround = p.completion - p.arrival
            p.waiting = p.turnaround - p.burst
            p.response = p.start_time - p.arrival
            gantt.append((p.pid, time, p.completion))
            time = p.completion
            done.append(p)
        return gantt, done

    def srtf(self):
        # (SRTF logic was correct, kept the time-step approach)
        procs = copy.deepcopy(self.processes)
        gantt, time, done = [], 0, []
        while len(done) < len(procs):
            avail = [p for p in procs if p.arrival <= time and p.remaining > 0]
            if not avail:
                # Handle idle time
                next_arrival = float('inf')
                for p in procs:
                    if p.remaining > 0 and p.arrival > time:
                        next_arrival = min(next_arrival, p.arrival)

                if next_arrival == float('inf'): # No more processes arriving
                    break

                if gantt and gantt[-1][0] == "Idle":
                    gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                else:
                    gantt.append(("Idle", time, next_arrival))
                time = next_arrival
                continue

            p = min(avail, key=lambda x: (x.remaining, x.arrival))

            if p.start_time == -1:
                p.start_time = time

            # Coalesce: Check if the last executed process was the same one
            if gantt and gantt[-1][0] == p.pid and gantt[-1][2] == time:
                gantt[-1] = (p.pid, gantt[-1][1], time + 1)
            else:
                gantt.append((p.pid, time, time + 1))

            p.remaining -= 1
            time += 1

            if p.remaining == 0:
                p.completion = time
                p.turnaround = p.completion - p.arrival
                p.waiting = p.turnaround - p.burst
                p.response = p.start_time - p.arrival
                done.append(p)
        return gantt, done

    def round_robin(self, q):
        procs = copy.deepcopy(self.processes)
        # Use a dictionary to map PID to the process object for easy lookup
        proc_map = {p.pid: p for p in procs}

        # Sort by arrival time
        sorted_procs = sorted(procs, key=lambda x: x.arrival)

        gantt, time, queue, done_count, idx = [], 0, [], 0, 0

        while done_count < len(procs):
            # Add newly arrived processes to the queue
            while idx < len(sorted_procs) and sorted_procs[idx].arrival <= time:
                if sorted_procs[idx].pid not in [p.pid for p in queue]:
                    queue.append(sorted_procs[idx])
                idx += 1

            if not queue:
                # Handle idle time
                if idx < len(sorted_procs):
                    next_arrival = sorted_procs[idx].arrival
                    if gantt and gantt[-1][0] == "Idle":
                        gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                    else:
                        gantt.append(("Idle", time, next_arrival))
                    time = next_arrival
                else: # All processes done, should not happen here if done_count check is correct
                    break
                continue

            p = queue.pop(0)

            if p.start_time == -1:
                p.start_time = time

            ex = min(q, p.remaining)

            # Add to Gantt
            gantt.append((p.pid, time, time + ex))
            p.remaining -= ex
            time += ex

            # Check for new arrivals during execution (Crucial RR step)
            while idx < len(sorted_procs) and sorted_procs[idx].arrival <= time:
                if sorted_procs[idx].pid not in [p_q.pid for p_q in queue] and sorted_procs[idx].remaining > 0:
                    queue.append(sorted_procs[idx])
                idx += 1

            if p.remaining > 0:
                # Put process back in queue after new arrivals
                queue.append(p)
            else:
                p.completion = time
                p.turnaround = p.completion - p.arrival
                p.waiting = p.turnaround - p.burst
                p.response = p.start_time - p.arrival
                done_count += 1

        return gantt, procs

    def priority_np(self):
        # (Priority NP logic was correct)
        procs = copy.deepcopy(self.processes)
        gantt, time, done = [], 0, []
        while len(done) < len(procs):
            # Sort by Priority (lower number is higher priority), then by Arrival Time
            avail = [p for p in procs if p.arrival <= time and p not in done]
            if not avail:
                next_arrival = min(p.arrival for p in procs if p not in done)
                if next_arrival > time:
                    if gantt and gantt[-1][0] == "Idle":
                        gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                    else:
                        gantt.append(("Idle", time, next_arrival))
                    time = next_arrival
                continue

            p = min(avail, key=lambda x: (x.priority, x.arrival))

            p.start_time = time
            p.completion = time + p.burst
            p.turnaround = p.completion - p.arrival
            p.waiting = p.turnaround - p.burst
            p.response = p.start_time - p.arrival
            gantt.append((p.pid, time, p.completion))
            time = p.completion
            done.append(p)
        return gantt, done

    def priority_p(self):
        # (Priority P logic was correct, kept the time-step approach)
        procs = copy.deepcopy(self.processes)
        gantt, time, done = [], 0, []
        while len(done) < len(procs):
            avail = [p for p in procs if p.arrival <= time and p.remaining > 0]
            if not avail:
                # Handle idle time
                next_arrival = float('inf')
                for p in procs:
                    if p.remaining > 0 and p.arrival > time:
                        next_arrival = min(next_arrival, p.arrival)

                if next_arrival == float('inf'):
                    break

                if gantt and gantt[-1][0] == "Idle":
                    gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                else:
                    gantt.append(("Idle", time, next_arrival))
                time = next_arrival
                continue

            # Select process with highest priority (lowest number), tie-break by arrival
            p = min(avail, key=lambda x: (x.priority, x.arrival))

            if p.start_time == -1:
                p.start_time = time

            # Coalesce
            if gantt and gantt[-1][0] == p.pid and gantt[-1][2] == time:
                gantt[-1] = (p.pid, gantt[-1][1], time + 1)
            else:
                gantt.append((p.pid, time, time + 1))

            p.remaining -= 1
            time += 1

            if p.remaining == 0:
                p.completion = time
                p.turnaround = p.completion - p.arrival
                p.waiting = p.turnaround - p.burst
                p.response = p.start_time - p.arrival
                done.append(p)
        return gantt, done

    def mlq(self):
        # FIX: The original MLQ implementation was flawed.
        # MLQ implies a strict hierarchy (Priority Q1 > Q2 > Q3).

        procs = copy.deepcopy(self.processes)
        done_procs = []
        gantt = []
        time = 0

        # Sort by arrival to know when they can start
        procs.sort(key=lambda x: x.arrival)

        while len(done_procs) < len(procs):
            # Find all arrived processes that are not yet done
            arrived_procs = [p for p in procs if p.arrival <= time and p not in done_procs]

            if not arrived_procs:
                # Handle idle time by jumping to the next arrival
                next_arrival = float('inf')
                for p in procs:
                    if p not in done_procs and p.arrival > time:
                        next_arrival = min(next_arrival, p.arrival)

                if next_arrival == float('inf'): break

                if gantt and gantt[-1][0] == "Idle":
                    gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                else:
                    gantt.append(("Idle", time, next_arrival))
                time = next_arrival
                continue

            # Separate into queues based on Priority (Q1: P=1, Q2: P=2, Q3: P>=3)
            q1_rr = [p for p in arrived_procs if p.priority == 1 and p.remaining > 0]
            q2_fcfs = [p for p in arrived_procs if p.priority == 2 and p.remaining > 0]
            q3_fcfs = [p for p in arrived_procs if p.priority >= 3 and p.remaining > 0]

            current_q, current_algo = None, None

            # Strict Priority Scheduling among queues: Q1 > Q2 > Q3
            if q1_rr:
                current_q = q1_rr
                current_algo = "RR"
            elif q2_fcfs:
                current_q = q2_fcfs
                current_algo = "FCFS"
                current_q.sort(key=lambda x: x.arrival) # Sort by arrival for FCFS
            elif q3_fcfs:
                current_q = q3_fcfs
                current_algo = "FCFS"
                current_q.sort(key=lambda x: x.arrival) # Sort by arrival for FCFS
            else:
                # This should not be reached if arrived_procs is non-empty, but as a safeguard
                time += 1
                continue

            p = current_q[0]

            if p.start_time == -1:
                p.start_time = time

            ex = 0
            if current_algo == "RR":
                q = 2 # Fixed quantum for RR queue
                ex = min(q, p.remaining)
            else: # FCFS (non-preemptive within the queue)
                ex = p.remaining

            # Find time of next event (arrival or higher priority arrival)
            next_event_time = float('inf')
            for proc in procs:
                if proc not in arrived_procs and proc.arrival > time:
                    next_event_time = min(next_event_time, proc.arrival)

            # In MLQ, higher-priority *queues* can preempt lower-priority queues.
            # If a process with P=1 (Q1) arrives during Q2/Q3 execution, Q2/Q3 is preempted.
            if current_algo != "RR":
                 # Check for preemption by a higher queue (P=1)
                preempt_time = min(p.arrival for p in procs if p.priority == 1 and p not in arrived_procs and p.arrival > time)
                if preempt_time < time + ex:
                    ex = preempt_time - time

            ex = min(ex, next_event_time - time)

            if ex <= 0: # Should not happen, but safe check
                time += 1
                continue

            gantt.append((p.pid, time, time + ex))
            p.remaining -= ex
            time += ex

            # Metrics update if completed
            if p.remaining == 0:
                p.completion = time
                p.turnaround = p.completion - p.arrival
                p.waiting = p.turnaround - p.burst
                p.response = p.start_time - p.arrival
                done_procs.append(p)

        # Only completed processes should be returned as metrics
        return gantt, [p for p in procs if p.remaining == 0]

    def mlfq(self, q1, q2, q3):
        # FIX: The original MLFQ logic was confusingly structured and had errors
        # in queue selection and demotion logic. This is a complete rewrite for correctness.

        # Use q1 as the base quantum. q2, q3 must be greater than q1.
        Q = {1: q1, 2: q2, 3: q3}

        procs = copy.deepcopy(self.processes)
        # Sort by arrival time
        procs.sort(key=lambda x: x.arrival)

        gantt, time, queue, done_count, idx = [], 0, [[], [], []], 0, 0

        while done_count < len(procs):
            # 1. Admission: New arrivals go to Q1
            while idx < len(procs) and procs[idx].arrival <= time:
                queue[0].append(procs[idx])
                idx += 1

            # 2. Selection: Find the highest priority non-empty queue
            current_q_idx = -1
            for i in range(3):
                if queue[i]:
                    current_q_idx = i
                    break

            if current_q_idx == -1:
                # Handle idle time
                if idx < len(procs):
                    next_arrival = procs[idx].arrival
                    if gantt and gantt[-1][0] == "Idle":
                        gantt[-1] = ("Idle", gantt[-1][1], next_arrival)
                    else:
                        gantt.append(("Idle", time, next_arrival))
                    time = next_arrival
                else:
                    break
                continue

            p = queue[current_q_idx].pop(0)
            current_q_level = current_q_idx + 1 # 1, 2, or 3
            q = Q[current_q_level]

            if p.start_time == -1:
                p.start_time = time

            ex = min(q, p.remaining)

            # Find the time of the next arrival that would enter Q1 (for preemption check)
            next_arrival_time = float('inf')
            if idx < len(procs):
                 next_arrival_time = procs[idx].arrival

            # Preemption: If a new process arrives in Q1 during Q2 or Q3 execution, preempt.
            if current_q_level > 1 and next_arrival_time < time + ex:
                ex = next_arrival_time - time

            if ex <= 0:
                # If execution time is zero (due to arrival check), put process back and advance time
                queue[current_q_idx].insert(0, p)
                time = next_arrival_time
                continue

            # Execute
            gantt.append((p.pid, time, time + ex))
            p.remaining -= ex
            p.service_time += ex
            time += ex

            # Check for new arrivals during execution again (post-execution admission)
            while idx < len(procs) and procs[idx].arrival <= time:
                queue[0].append(procs[idx])
                idx += 1

            if p.remaining == 0:
                # Completion
                p.completion = time
                p.turnaround = p.completion - p.arrival
                p.waiting = p.turnaround - p.burst
                p.response = p.start_time - p.arrival
                done_count += 1
                p.current_queue = -1 # Mark as done
            else:
                # Demotion/Feedback: If consumed full quantum, move to next queue (if not Q3)
                if ex == q and current_q_level < 3:
                    p.current_queue += 1
                    queue[current_q_level].append(p) # current_q_level is index + 1
                # If preempted by a new arrival (ex < q) or is in Q3 (FCFS), stay in current queue
                else:
                    queue[current_q_idx].append(p)

        # Only completed processes should be returned as metrics
        return gantt, [p for p in procs if p.remaining == 0]

    # --- RESULTS DISPLAY ---

    def display_results(self, gantt, metrics, algo):
        # (Results display logic was robust and remains the same)
        for w in self.right_panel.winfo_children():
            w.destroy()

        tk.Label(self.right_panel, text=f"📊 Results - {algo}",
                 font=("Helvetica", 18, "bold"), bg="#334155", fg="white").pack(pady=10)

        # Gantt Chart
        gantt_frame = tk.Frame(self.right_panel, bg="#1e293b", relief=tk.RAISED, bd=2)
        gantt_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        tk.Label(gantt_frame, text="📈 Gantt Chart", font=("Helvetica", 14, "bold"),
                 bg="#1e293b", fg="white").pack(pady=5)

        fig = Figure(figsize=(12, 2), facecolor="#1e293b")
        ax = fig.add_subplot(111)
        ax.set_facecolor("#1e293b")

        # Filter out "Idle" segments if they exist for coloring, but include them in the chart
        pid_colors = {}
        for i, p in enumerate(self.processes):
             pid_colors[p.pid] = self.colors[i % len(self.colors)]

        for pid, start, end in gantt:
            color = pid_colors.get(pid, '#64748b') # Grey for Idle
            ax.barh(0, end - start, left=start, height=0.5,
                    color=color, edgecolor='white', linewidth=2)
            if pid != "Idle":
                ax.text((start + end) / 2, 0, pid, ha='center', va='center',
                        color='white', fontweight='bold', fontsize=10)

        ax.set_ylim(-0.5, 0.5)
        max_time = gantt[-1][2] if gantt and gantt[-1] else 10
        ax.set_xlim(0, max_time + 1) # +1 for visual spacing
        ax.set_xlabel('Time', color='white', fontsize=12)
        ax.set_yticks([])
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_color('white')
        ax.tick_params(colors='white')
        ax.grid(True, axis='x', alpha=0.3, color='white')

        canvas = FigureCanvasTkAgg(fig, gantt_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        # Metrics
        metrics_frame = tk.Frame(self.right_panel, bg="#334155", relief=tk.RAISED, bd=2)
        metrics_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        tk.Label(metrics_frame, text="📊 Performance Metrics",
                 font=("Helvetica", 14, "bold"), bg="#334155", fg="white").pack(pady=5)

        # Calculate averages safely
        if not metrics:
            avg_tat, avg_wt, avg_rt = 0.0, 0.0, 0.0
        else:
            avg_tat = sum(p.turnaround for p in metrics) / len(metrics)
            avg_wt = sum(p.waiting for p in metrics) / len(metrics)
            avg_rt = sum(p.response for p in metrics) / len(metrics)

        avg_frame = tk.Frame(metrics_frame, bg="#334155")
        avg_frame.pack(fill=tk.X, padx=10, pady=10)

        for label, val, color in [("Avg TAT", f"{avg_tat:.2f}", "#3b82f6"),
                                   ("Avg WT", f"{avg_wt:.2f}", "#10b981"),
                                   ("Avg RT", f"{avg_rt:.2f}", "#8b5cf6")]:
            box = tk.Frame(avg_frame, bg=color, relief=tk.RAISED, bd=2)
            box.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5)
            tk.Label(box, text=label, bg=color, fg="white",
                     font=("Helvetica", 10)).pack(pady=5)
            tk.Label(box, text=val, bg=color, fg="white",
                     font=("Helvetica", 18, "bold")).pack(pady=5)

        # Table
        table = tk.Frame(metrics_frame, bg="#334155")
        table.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        tree = ttk.Treeview(table, columns=("PID", "AT", "BT", "CT", "TAT", "WT", "RT"),
                            show="headings", height=10)
        for col in ["PID", "AT", "BT", "CT", "TAT", "WT", "RT"]:
            tree.heading(col, text=col)
            tree.column(col, width=70, anchor=tk.CENTER)

        for p in sorted(metrics, key=lambda x: x.pid):
            tree.insert("", tk.END, values=(p.pid, p.arrival, p.burst, p.completion,
                                            p.turnaround, p.waiting, p.response))

        scroll = ttk.Scrollbar(table, orient=tk.VERTICAL, command=tree.yview)
        tree.configure(yscrollcommand=scroll.set)
        tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scroll.pack(side=tk.RIGHT, fill=tk.Y)

def main():
    root = tk.Tk()
    # Apply a modern style to the Treeview widgets
    style = ttk.Style()
    style.theme_use("clam") # Use a theme that supports customization
    style.configure("Treeview", background="#334155",
                    foreground="white", fieldbackground="#334155",
                    font=('Helvetica', 10), rowheight=25)
    style.map('Treeview', background=[('selected', '#3b82f6')])
    style.configure("Treeview.Heading", background="#1e293b",
                    foreground="#94a3b8", font=('Helvetica', 10, 'bold'))

    app = CPUScheduler(root)
    root.mainloop()

if __name__ == "__main__":
    main()

TclError: no display name and no $DISPLAY environment variable